# Datamine LOADCF Process Example

This notebook demonstrates how to configure and run the **`loadcf`** process wrapper in `dmstudio`.

## Process Description

## LOADCF

Convert a macro file (*.mac) into a standard menu file (*.men) for faster execution.

LOADCF pre-processes macros to increase their general runtime performance. Specific optimization can be performed for customized SCREEN based menu applications. Three levels of processing (controlled by setting parameter LEVEL in the dialog) are available:

LEVEL = 0 Converts the macro file into the standard .STK and .MEN files.

LEVEL = 1 Optimizes the performance for screen based menus with the following:

1. Pre-processes and verifies !SCREEN definitions to create 'tokenized' SCreen Code (.SCC) and SCreen Text (.SCT) files for the menu.
2. Replaces !SCREEN code with references to relevant entries in .SCC file.
3. Strips !REM and # inline remarks, remove indentation.

**Note** : If there is more than one macro in the file to be loaded, only the first macro will be loaded. .

You can choose the name of the binary random access menu file to be created. This may be up to 56 characters long, and may include pathnames. It must not contain the character '.', except as part of an up to three character extension. If an extension is given, it will be used; otherwise an extension of form .MEN will be appended. The following are legal menu names:-

* mymenu.men

* Datamine/mymenu.dat

If the response to the menu name question is blank<return> or just <return>, then a name MENUFILE.DAT will be assumed. This is for compatibility with earlier releases of Datamine products.

The menu name set up in !**LOADCF** is the one that must be specified in the !**menu** or !**nomenu** commands.

* **Note** (The !**LOADCF** process deletes the stack file <name>.STK, where <name> is the requested macro name minus the extension.):

### Input Files:

### Output Files:

### Fields:

### Parameters:

* **print**:
  Macro line display (0). =0 Do not display.=1 Display each line of the macro file as
  loaded.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **level**:
  Level of menu compilation . =0 Standard compilation. =1 'Optimise' for !SCREEN
  processing (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **encrypt**:
  Encryption level (0). =0 None. =1 Macro is encrypted
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('loadcf')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute loadcf
print("Running loadcf...")
dm_fil.loadcf(
    # print_p=0,  # optional
    # level_p=0,  # optional
    # encrypt_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("loadcf execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_loadcf_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")